# AgentO11y Getting Started Notebook

This notebook is a hands-on tour of the **`agento11y` (Observability) SDK** in
`teradata_agentstack`: how to **observe, trace, evaluate, and measure** an AI agent —
projects, traces & spans, evaluation rules, online/offline evaluation, feedback scores, and
the metrics dashboard.

Observability needs a **real agent producing real traces**, so the notebook first uses the
**`agentops` (Core) SDK** as a **prerequisite** to build, deploy, invoke, and add memory to a
small, Opik-instrumented agent (Sections 3–6). That agent is simply the *subject* of the
observability work that follows. **Already have an agent emitting traces?** Jump straight to
[Section 7](#7-observability-projects).

## Why This Matters
- The focus is **AgentO11y**: once an agent is running, a single SDK gives you traces, spans,
  automated LLM-judge evaluation, human feedback, datasets, and aggregate metrics.
- A **single JWT bearer token authenticates both services** — you provide it securely when
  prompted.
- Enter your connection details once (you are prompted for only `BASE_URL`, the token, and an
  optional model key), then reuse them across every section.
- The AgentOps prerequisite (Sections 3–6) builds, deploys, and invokes that agent so the
  observability sections have real traces to work with.

## What You Will Accomplish
**With AgentO11y — the focus (Sections 7–16):**
- Discover observability **projects** and select the one your agent writes to.
- Search and inspect **traces** and **spans** (the LLM/tool steps inside each run).
- Create **evaluation rules** and run **online** and **offline** evaluation.
- Record **feedback scores**, browse the **metrics catalog**, and read the **metrics dashboard**.

**With AgentOps — prerequisite to create something to observe (Sections 3–6, 17):**
- Build, deploy, invoke, and add memory to a small instrumented agent, then optionally retire it.

## Table of Contents

- [**1. Setup & Prerequisites**](#1-setup--prerequisites)
  - [1.1 Import Required Modules](#11-import-required-modules)
  - [1.2 Authentication](#12-authentication)
  - [1.3 Provide Connection Details](#13-provide-connection-details)
  - [1.4 Initialize Required Classes](#14-initialize-required-classes)
  - [1.5 Demo Configuration & Runtime Holders](#15-demo-configuration--runtime-holders)
  - [1.6 Verify Authentication](#16-verify-authentication)
- [**2. Health Checks**](#2-health-checks)
- [**3. Build Your Agent**](#3-build-your-agent--create--validate-the-agent-definition) *(AgentOps prerequisite)*
  - [3.1 Create the Agent Definition](#31-create-the-agent-definition-from-the-zip)
  - [3.2 Validate the Agent Definition](#32-validate-the-agent-definition)
- [**4. Deploy Your Agent**](#4-deploy-your-agent) *(AgentOps prerequisite)*
  - [4.1 Create the Deployment](#41-create-the-deployment)
  - [4.2 Wait & Inspect the Deployment](#42-wait-for-the-deployment-to-become-active-then-inspect-it)
- [**5. Invoke Your Agent**](#5-invoke-your-agent) *(AgentOps prerequisite)*
- [**6. Add Memory**](#6-add-memory--session-memory-test) *(AgentOps prerequisite)*
- [**7. Observability Projects**](#7-observability-projects)
- [**8. (Optional) Generate Additional Traces**](#8-optional-generate-additional-traces)
- [**9. Search & View Traces**](#9-search--view-traces)
  - [9b. List All Spans for a Project](#9b-list-all-spans-for-a-project)
- [**10. Evaluation Rules**](#10-evaluation-rules)
- [**11. Trigger Online Evaluation**](#11-trigger-online-evaluation)
- [**12. Feedback Scores**](#12-feedback-scores)
- [**13. Metrics Catalog**](#13-metrics-catalog)
- [**14. Dataset Management**](#14-dataset-management)
- [**15. Offline Evaluation**](#15-offline-evaluation)
- [**16. Metrics Dashboard**](#16-metrics-dashboard)
- [**17. (Optional) Retire the Agent**](#17-optional-retire-the-agent) *(AgentOps prerequisite)*
- [**18. Summary & Next Steps**](#18-summary--next-steps)
- [**19. Data Models Reference**](#19-data-models-reference)

## 1. Setup & Prerequisites

This section prepares everything you need: imports, authentication, connection details, the initialized clients, and the demo configuration.

### 1.1 Import Required Modules

Run this first. It imports the AgentOps **Core** and **O11y** classes and request models, prints the SDK surface with `blueprint()`, and defines the `show_output()` / `to_dict()` helpers used throughout.


In [ ]:
import json
import os
import time
import uuid
from getpass import getpass
from pprint import pprint

import requests

from teradata_agentstack import BearerAuth

# ── AgentOps O11y (Observability) SDK — observe, trace, and evaluate ─────────
from teradata_agentstack.agento11y import (
    AgentO11yClient,
    Health,
    Projects,
    Traces,
    Spans,
    Evaluations,
    Datasets,
    FeedbackScores,
    Metrics,
    blueprint,
    # Enums — use these instead of raw strings for type-safe calls
    SpanType,
    TraceStatus,
    EvalCategory,
    EvalType,
    ScoreType,
    TaskType,
)
from teradata_agentstack.agento11y.models import (
    EvaluationRuleCreate,
    VariableMapping,
    ScoreDefinition,
    EvalJobRequest,
    FeedbackScoreBatchRequest,
    FeedbackScoreItem,
    FeedbackScore,
    OfflineEvalRequest,
    OfflineMetric,
    DatasetCreate,
)

# ── AgentOps Core SDK — build, deploy, and operate the agent ─────────────────
# Same installed package; aliased where names collide with the O11y SDK above.
from teradata_agentstack.agentops import (
    AgentOpsClient,
    Definitions,
    Deployments,
    Framework,
    Health as CoreHealth,
)
from teradata_agentstack.agentops.models import (
    AgentDefinitionCreateRequest,
    ArtifactsConfig,
    DeploymentCreateRequest,
    DeploymentEngineTypeConfig,
    DeploymentResourceConfig,
    DeploymentServiceConfig,
)

**Print the SDK surface.** `blueprint()` lists every O11y client, resource group, and request model the SDK exposes — a quick map of what you can call.

In [1]:
blueprint()

----------------------------------------------------------------
Available classes for AgentO11y SDK:
    * teradata_agentstack.agento11y.Health
    * teradata_agentstack.agento11y.Projects
    * teradata_agentstack.agento11y.Traces
    * teradata_agentstack.agento11y.Spans
    * teradata_agentstack.agento11y.Evaluations
    * teradata_agentstack.agento11y.Datasets
    * teradata_agentstack.agento11y.FeedbackScores
    * teradata_agentstack.agento11y.Metrics
----------------------------------------------------------------


**Define two display helpers.** `show_output()` pretty-prints any SDK response, and `to_dict()` converts a response object into a plain dictionary. Both are reused throughout the notebook.

In [ ]:
def show_output(label, value):
    """Pretty-print SDK responses."""
    print(f"\n{label}:")
    try:
        if hasattr(value, "model_dump") and callable(value.model_dump):
            pprint(value.model_dump(), sort_dicts=False, width=120)
        elif hasattr(value, "dict") and callable(value.dict):
            pprint(value.dict(), sort_dicts=False, width=120)
        else:
            pprint(value, sort_dicts=False, width=120)
    except Exception:
        pprint(value, sort_dicts=False, width=120)


def to_dict(obj):
    """Convert SDK response to plain dict for attribute access."""
    if isinstance(obj, dict):
        return obj
    if hasattr(obj, "model_dump"):
        return obj.model_dump()
    return vars(obj)

### 1.2 Authentication

The `teradata_agentstack` clients support **4 authentication modes** and **3 ways to provide credentials**. A **single** Bearer token authenticates **both** the Core and O11y services.

The credential sources are resolved in this order: direct `auth` parameter, environment variables, then YAML config file.

#### Authentication Modes

| Auth Mode | Class | Required Fields |
| --- | --- | --- |
| Bearer Token | `BearerAuth` | `auth_bearer` |
| Basic Auth | `BasicAuth` | `username`, `password` |
| Client Credentials (OAuth2) | `ClientCredentialsAuth` | `auth_token_url`, `auth_client_id`, `auth_client_secret` |
| Device Code (OAuth2) | `DeviceCodeAuth` | `auth_token_url`, `auth_client_id`, `auth_client_secret`, `auth_device_auth_url` |

This notebook uses `BearerAuth` for both services. You enter the token securely via `getpass` in step 1.3 — it is never hardcoded.

### 1.3 Provide Connection Details

Enter the AgentOps service URL when prompted &mdash; a single `BASE_URL` fronts **both** the Core and O11y services, so it is usually the only value you must type. This cell also captures:

- **AUTH_TOKEN** &mdash; entered securely via `getpass` so it is never echoed or stored in the notebook. A single token authenticates **both** services.
- **OPENAI_API_KEY** &mdash; optional; injected into the deployment runtime (Section 4) and used for LLM-based evaluations. Press Enter to skip.
- **SSL_VERIFY** &mdash; defaults to `False` for demo gateways with self-signed certificates. Set `True` when your environment uses trusted certificates.

In [2]:
# ── Connection credentials — entered at runtime, never hardcoded ─────────────
# Root URL of the AgentOps service — a single BASE_URL fronts BOTH the Core
# (build / deploy / invoke) and O11y (observe / evaluate) services.
BASE_URL = getpass("Enter BASE_URL (AgentOps API): ").strip()

# Session token (JWT). A SINGLE token authenticates BOTH the Core and O11y services,
# entered securely. Tokens are short-lived (~2 hours) — an expired token causes
# 401 "Invalid token".
AUTH_TOKEN = getpass("Enter AUTH_TOKEN: ").strip()

# Optional: injected into the deployment runtime config (Section 4) so the deployed
# agent can call the model. Press Enter to skip.
OPENAI_API_KEY = getpass("Enter OPENAI_API_KEY (optional, press Enter to skip): ").strip()
if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# Verify the endpoint's TLS certificate? Defaults to False for self-signed dev gateways.
SSL_VERIFY = False

print("Credentials ready.")
print(f"BASE_URL configured:    {bool(BASE_URL)}")
print(f"AUTH_TOKEN configured:  {bool(AUTH_TOKEN)}")
print(f"OPENAI_API_KEY set:     {bool(os.environ.get('OPENAI_API_KEY'))}")
print(f"SSL_VERIFY:             {SSL_VERIFY}")

Credentials ready.
BASE_URL configured:    True
AUTH_TOKEN configured:  True
OPENAI_API_KEY set:     True
SSL_VERIFY:             False


### 1.4 Initialize Required Classes

This creates the authenticated **O11y** client (`AgentO11yClient`) and **Core** client (`AgentOpsClient`) from a single `BearerAuth`, then the helper classes (`Health`, `Projects`, `Traces`, `Spans`, `Evaluations`, `Datasets`, `FeedbackScores`, `Metrics`, plus Core `Definitions` / `Deployments`) so the later examples can run directly.


In [3]:
# Initialize the SDK clients using the credentials configured above.
# A single BearerAuth (JWT) authenticates BOTH the Core and O11y services.
auth = BearerAuth(auth_bearer=AUTH_TOKEN)

# ── O11y (Observability) client + resource objects ──────────────────────────
client = AgentO11yClient(
    base_url=BASE_URL,
    auth=auth,
    ssl_verify=SSL_VERIFY,
    project_id=None,  # set after discovering projects (Section 7)
)
health_api = Health(client=client)
projects_api = Projects(client=client)
traces_api = Traces(client=client)
spans_api = Spans(client=client)
evaluations_api = Evaluations(client=client)
datasets_api = Datasets(client=client)
feedback_api = FeedbackScores(client=client)
metrics_api = Metrics(client=client)
print("AgentOps O11y SDK initialized")

# ── Core (build / deploy / invoke / memory) client + resource objects ────────
# Same BASE_URL as the O11y client — one gateway fronts both services.
core_client = AgentOpsClient(base_url=BASE_URL, auth=auth, ssl_verify=SSL_VERIFY)
agent_definitions = Definitions(client=core_client)
deployments = Deployments(client=core_client)
core_health_api = CoreHealth(client=core_client)
print("AgentOps Core SDK initialized")
print(f"  Base URL : {BASE_URL}")
print(f"  OPENAI_API_KEY set: {bool(os.environ.get('OPENAI_API_KEY'))}")

AgentOps O11y SDK initialized
AgentOps Core SDK initialized
  Base URL : <internal-host>
  OPENAI_API_KEY set: True


### 1.5 Demo Configuration & Runtime Holders

These are **editable constants** that drive the agent build → deploy → invoke flow
(Sections 3–6). Unlike credentials, they are not prompted — change a value here and
the step that uses it picks it up.

- **Agent identity** — name, description, framework, tags, and `ZIP_PATH`
  (`~/agent_opik_v3.zip`, an Opik-instrumented, memory-enabled LangGraph agent on the
  kernel's filesystem).
- **Deployment settings** — name, namespace, image, resources, and `RUNTIME_CONFIG`
  (the environment passed to the agent container).
- **Runtime holders** — `AGENT_DEF_ID`, `DEPLOYMENT_ID`, and `agent_url` start as
  `None` and are filled in as the notebook runs; later sections and cleanup consume them.
  Re-running this cell resets them.

> Defaults are demo-ready: you normally only adjust `ZIP_PATH` to point at your own
> agent archive before running.


In [4]:
# ── Agent identity (used by Sections 3-4) ───────────────────────────────────
WORKSPACE_ID = None  # None = default workspace
_run_suffix = uuid.uuid4().hex[:8]  # keeps names unique across re-runs
AGENT_DEFINITION_NAME = f"agentops-o11y-demo-agent-{_run_suffix}"
AGENT_DEFINITION_DESCRIPTION = "Opik-instrumented LangGraph agent for the AgentOps O11y end-to-end demo"
AGENT_FRAMEWORK = Framework.langgraph
AGENT_TAGS = ["langgraph", "opik", "observability", "demo"]

# ZIP archive with the agent code, in the kernel user's home directory.
ZIP_PATH = os.path.expanduser("~/agent_opik_v3.zip")

# ── Deployment settings (used by Section 4) ─────────────────────────────────
DEPLOYMENT_NAME = f"agentops-o11y-demo-{_run_suffix}"
DEPLOYMENT_NAMESPACE = "agentruntime"
DEPLOYMENT_ENVIRONMENT = "dev"
# Container image that runs the agent — your platform administrator provides this.
DOCKER_IMAGE = "<docker-registry>/<image-path>/vmo-python-base:3.11.5"
ENGINE_TYPE = "DOCKER_RESTFUL"
REPLICAS = 1
NODE_TYPE = "default-wl"
MEMORY = "2Gi"
CPU = "1000m"
SERVICE_PORT = 5000

# Env vars passed to the agent container; OPENAI_API_KEY (if set) is added below.
RUNTIME_CONFIG = {
    "ENFORCE_AUTHORIZATION": "true",
    "OPENAI_MODEL_NAME": "us-anthropic-claude-sonnet-4-5-20250929-v1-0-profile",
    "OPENAI_EMBEDDING_MODEL_NAME": "amazon-titan-embed-text-v2-0-profile",
    # Model-hub endpoint — your platform administrator provides this.
    "TD_AI_STUDIO_AI_MODELHUB": "http://<model-hub-host>:4000",
}
if OPENAI_API_KEY:
    RUNTIME_CONFIG["OPENAI_API_KEY"] = OPENAI_API_KEY

# ── Mutable state populated as the notebook runs ────────────────────────────
AGENT_DEF_ID = None      # set in Section 3
DEPLOYMENT_ID = None     # set in Section 4 (also the observability project name)
agent_url = None         # set in Section 4 (used by Sections 5-6)

print("Demo configuration loaded.")
print(f"  Agent name      : {AGENT_DEFINITION_NAME}")
print(f"  Deployment name : {DEPLOYMENT_NAME}")
print(f"  ZIP_PATH        : {ZIP_PATH}")
print(f"  ZIP exists      : {os.path.exists(ZIP_PATH)}")
print(f"  DOCKER_IMAGE    : {DOCKER_IMAGE}")

Demo configuration loaded.
  Agent name      : agentops-o11y-demo-agent-b78112df
  Deployment name : agentops-o11y-demo-b78112df
  ZIP_PATH        : /root/agent_opik_v3.zip
  ZIP exists      : True
  DOCKER_IMAGE    : <redacted-host>


### 1.6 Verify Authentication

If auth is enabled, confirm the token works against the health endpoint
(which is excluded from auth) and a protected endpoint.


In [5]:
# Health endpoint is public
h = health_api.health_check()
print("Health (no auth needed) : OK")
show_output("Health", h)

# Projects endpoint requires auth
try:
    projects_api.list()
    print("Projects (with token)   : OK")
except Exception as e:
    print(f"Projects (with token)   : FAILED - {e}")

Health (no auth needed) : OK

Health:
{'status': 'healthy', 'version': '0.1.0', 'timestamp': '2026-06-22T17:37:43.373633Z'}
Projects (with token)   : OK


## 2. Health Checks

Confirm the service is up before doing real work: the code cell calls the liveness and readiness endpoints (both are public, so they need no token).

**SDK:** `health_api.health_check()` / `health_api.readiness()`


In [6]:
# Liveness probe (public, no token)
try:
    liveness = health_api.health_check()
    print("=== Liveness Check ===")
    show_output("Liveness", liveness)
except Exception as e:
    print(f"Liveness check failed: {e}")

print()

# Readiness probe (public, no token)
try:
    readiness = health_api.readiness()
    print("=== Readiness Check ===")
    show_output("Readiness", readiness)
except Exception as e:
    print(f"Readiness check failed: {e}")


=== Liveness Check ===

Liveness:
{'status': 'healthy', 'version': '0.1.0', 'timestamp': '2026-06-22T17:37:46.161275Z'}

=== Readiness Check ===

Readiness:
{'status': 'ready',
 'checks': {'opik': {'status': 'connected',
                     'url': '<internal-host>',
                     'latency_ms': 52,
                     'error': None}}}


---

## Prerequisite — Create an Agent to Observe (AgentOps Core SDK)

> **Sections 3–6 are setup, not the focus.** They use the **`agentops` Core SDK** to build, deploy, invoke, and add memory to a small, Opik-instrumented agent — so the observability sections that follow have a *real* agent producing *real* traces.
>
> **Already have an agent emitting traces?** Skip straight to [Section 7 — Observability Projects](#7-observability-projects) and point the project selector at it. Otherwise, run these four sections once to stand up the demo agent.


## 3. Build Your Agent — Create & Validate the Agent Definition

> **Needs the agent ZIP** on the kernel's filesystem at `ZIP_PATH` (`~/agent_opik_v3.zip`).

An **agent definition** registers your agent's code, framework, and configuration with
the AgentOps Core service. It is the first step of the lifecycle and the thing you later
*deploy*. Here we register the definition straight from the local ZIP archive
(`~/agent_opik_v3.zip`) — an Opik-instrumented, memory-enabled LangGraph agent — using
the typed models `AgentDefinitionCreateRequest` + `ArtifactsConfig`.

**SDK:** `agent_definitions.create(body=..., file=ZIP_PATH)` → `agent_definitions.poll(id=...)`

### 3.1 Create the agent definition from the ZIP

`create()` uploads the archive and returns a definition whose `.id` we keep as
`AGENT_DEF_ID`; `poll()` then blocks until the artifacts finish processing (status
`published`).


In [7]:
# Register the agent definition straight from the local ZIP archive.
try:
    create_request = AgentDefinitionCreateRequest(
        name=AGENT_DEFINITION_NAME,
        description=AGENT_DEFINITION_DESCRIPTION,
        framework=AGENT_FRAMEWORK,
        artifacts=ArtifactsConfig(storage_type="zip"),
        tags=AGENT_TAGS,
        framework_template="passthrough",
        workspace_id=WORKSPACE_ID,
    )
    created = agent_definitions.create(body=create_request, file=ZIP_PATH)
    show_output("Created Agent Definition", created)

    AGENT_DEF_ID = getattr(created, "id", None)
    # Block until the uploaded artifacts finish processing (status -> published).
    agent_definitions.poll(id=AGENT_DEF_ID)
    print(f"\nAGENT_DEF_ID: {AGENT_DEF_ID}")
except Exception as e:
    print(f"Create agent definition failed: {e}")


Created Agent Definition:
{'id': 'fff652ad-ff26-4da4-a749-8bfd4ca75afc',
 'name': 'agentops-o11y-demo-agent-b78112df',
 'status': 'creating',
 'storage_type': 'zip',
 'latest_version': {'id': '785ed072-f57f-4ee3-888c-b3420331e151',
                    'agent_definition_id': 'fff652ad-ff26-4da4-a749-8bfd4ca75afc',
                    'version_number': 1,
                    'status': 'creating',
                    'version_notes': None,
                    'artifacts_path': None,
                    'error_message': None,
                    'created_at': '2026-06-22T17:37:52.823884Z',
                    'updated_at': '2026-06-22T17:37:52.823888Z'},
 'active_deployment_count': 0,
 'created_at': '2026-06-22T17:37:52.816396Z'}
Polling Definitions (every 5s, max 20 attempts)...
  [1/20] Status: creating
  [2/20] Status: published
  Done — status: published

AGENT_DEF_ID: fff652ad-ff26-4da4-a749-8bfd4ca75afc


### 3.2 Validate the agent definition

Confirm the definition is registered and ready before deploying it. `get()` returns the
full metadata and current status; `list()` shows it alongside any other definitions in
the workspace.

**SDK:** `agent_definitions.get(id=...)` / `agent_definitions.list(...)`


In [ ]:
try:
    # Full metadata + current status of the definition we just created.
    show_output("Get Agent Definition", agent_definitions.get(id=AGENT_DEF_ID))

    # Confirm it appears in the workspace listing.
    definitions_response = agent_definitions.list(
        workspace_id=WORKSPACE_ID,
        framework=AGENT_FRAMEWORK,
        name=None,
        description=None,
        include_archived=False,
        include_failed=False,
    )
    show_output("List Agent Definitions", definitions_response)
except Exception as e:
    print(f"Validate agent definition failed: {e}")

## 4. Deploy Your Agent

A **deployment** is a running instance of the agent definition on Kubernetes. We deploy
with two important flags:

- **`enable_memory=True`** — the platform provisions a PostgreSQL database and injects
  `MEMORY_DB_URI` into the agent pod. This agent *requires* it (it builds a checkpointer
  from `MEMORY_DB_URI` at startup), and it is what makes the Section 6 memory test work.
- **`in_house_observability=True`** — wires the agent's Opik traces to the O11y service
  and auto-creates an observability **project named after the deployment id**, which
  Section 7 then selects so the rest of the notebook inspects *these* real traces.

**SDK:** `deployments.create(body=DeploymentCreateRequest(...))`

### 4.1 Create the deployment

The whole request is built from typed models. `OPENAI_API_KEY` (if you entered one)
is passed through `RUNTIME_CONFIG`. The returned `agent_id` is the deployment id we keep
as `DEPLOYMENT_ID`.


In [8]:
try:
    created_dep = deployments.create(
        body=DeploymentCreateRequest(
            agent_definition_id=AGENT_DEF_ID,
            deployment_name=DEPLOYMENT_NAME,
            namespace=DEPLOYMENT_NAMESPACE,
            environment=DEPLOYMENT_ENVIRONMENT,
            version_identifier=None,
            engine_type=ENGINE_TYPE,
            enable_memory=True,           # provisions PostgreSQL + injects MEMORY_DB_URI
            in_house_observability=True,  # routes the agent's Opik traces to the O11y service
            runtime_config=RUNTIME_CONFIG,
            runtime_secrets_config=None,
            engine_type_config=DeploymentEngineTypeConfig(
                docker_image=DOCKER_IMAGE,
                replicas=REPLICAS,
                node_type=NODE_TYPE,
                resources=DeploymentResourceConfig(memory=MEMORY, cpu=CPU),
                service=DeploymentServiceConfig(type="ClusterIP", port=SERVICE_PORT, target_port=SERVICE_PORT),
            ),
        )
    )
    show_output("Created Deployment", created_dep)
    DEPLOYMENT_ID = getattr(created_dep, "agent_id", None) or getattr(created_dep, "id", None)
    print(f"\nDEPLOYMENT_ID: {DEPLOYMENT_ID}")
except Exception as e:
    print(f"Create deployment failed: {e}")


Created Deployment:
{'in_house_observability': True,
 'agent_id': '01e67f83-d5d5-4510-9eaa-534258272ee0',
 'agent_definition_id': 'fff652ad-ff26-4da4-a749-8bfd4ca75afc',
 'agent_definition_name': 'agentops-o11y-demo-agent-b78112df',
 'workspace_id': 'f0b8a454-99d7-474f-8ec0-00094c1ab6a0',
 'deployment_name': 'agentops-o11y-demo-b78112df',
 'environment': 'dev',
 'namespace': 'agentruntime',
 'version_id': '785ed072-f57f-4ee3-888c-b3420331e151',
 'version_number': 1,
 'deployment_history': [{'user': '<user>',
                         'record_id': 'd877a0c8-cb26-4398-879c-90ef343bab41',
                         'timestamp': '2026-06-22T17:38:07.353168+00:00',
                         'version_id': '785ed072-f57f-4ee3-888c-b3420331e151',
                         'previous_version_id': None}],
 'runtime_config': {'OPENAI_API_KEY': 'sk-REDACTED',
                    'OPENAI_MODEL_NAME': 'us-anthropic-claude-sonnet-4-5-20250929-v1-0-profile',
                    'OPIK_PROJECT_NAME': '01e67f

### 4.2 Wait for the deployment to become active, then inspect it

`poll()` blocks until the deployment reaches a terminal state (`active`, `failed`, or
`stopped`). Once active, `get()` exposes the runtime details — most importantly the
`agent_url` the next sections call — and `logs()` returns the most recent container logs.

**SDK:** `deployments.poll(id=...)` / `deployments.get(id=...)` / `deployments.logs(id=...)`


In [ ]:
try:
    # Block until the deployment reaches a terminal state (active / failed / stopped).
    deployments.poll(id=DEPLOYMENT_ID)

    # Capture runtime details, including the URL used to invoke the agent.
    deployment = deployments.get(id=DEPLOYMENT_ID)
    show_output("Deployment Details", deployment)
    agent_url = getattr(deployment, "agent_url", None)
    print(f"\nStatus    : {getattr(deployment, 'status', None)}")
    print(f"Agent URL : {agent_url}")

    # Most recent logs from the running container.
    logs = deployments.logs(id=DEPLOYMENT_ID, tail_lines=200, container=None, follow=False)
    show_output("Deployment Logs (last 200 lines)", logs)
except Exception as e:
    print(f"Validate deployment failed: {e}")

## 5. Invoke Your Agent

With the agent running, call it directly over HTTP to confirm it is healthy and to
**generate real traces** that the observability sections will analyze. We hit three
endpoints on the captured `agent_url`, authenticating with the same JWT:

| Endpoint | Purpose |
|---|---|
| `GET /health` | Liveness of the agent container |
| `GET /info` | Agent metadata (name, framework, capabilities) |
| `POST /invoke` | Send a natural-language message and get the agent's answer |

The invoke body matches this agent's contract — `{"input": {"message": ...}, "streaming": False}` —
and the answer is returned under `response.json()["output"]`.


In [ ]:
invoke_headers = {"Authorization": f"Bearer {AUTH_TOKEN}", "Content-Type": "application/json"}
try:
    # Liveness of the deployed agent itself (separate from the control-plane health in Section 2).
    r = requests.get(f"{agent_url}/health", headers=invoke_headers, timeout=30, verify=SSL_VERIFY)
    print(f"/health → {r.status_code}")
    print(json.dumps(r.json(), indent=2) if r.ok else r.text[:500])

    # Agent metadata (name, framework, capabilities).
    r = requests.get(f"{agent_url}/info", headers=invoke_headers, timeout=30, verify=SSL_VERIFY)
    print(f"\n/info → {r.status_code}")
    print(json.dumps(r.json(), indent=2) if r.ok else r.text[:500])

    # Send a few messages — each one produces a trace (root span + LLM/tool child spans).
    demo_messages = [
        "List my databases",
        "What can you help me with?",
    ]
    for msg in demo_messages:
        payload = {"input": {"message": msg}, "streaming": False}
        r = requests.post(f"{agent_url}/invoke", headers=invoke_headers, json=payload, timeout=120, verify=SSL_VERIFY)
        print(f"\n/invoke ({msg!r}) → {r.status_code}")
        if r.ok:
            body = r.json()
            print(json.dumps(body.get("output", body), indent=2, default=str)[:1500])
        else:
            print(r.text[:500])
    print("\nInvocations complete — traces should now be visible in the observability project (Sections 7+).")
except Exception as e:
    print(f"Invoke agent failed: {e}")

## 6. Add Memory — Session Memory Test

Because we deployed with **`enable_memory=True`**, the agent persists conversation state
per `session_id`. To prove it, we send two messages on the **same** `session_id`: the
second question refers back to the first ("the first one") without restating it. If the
agent answers in context, session memory is working.

| Turn | Message |
|---|---|
| 1 | "List my databases" |
| 2 | "Can you tell me more about the first one?" |


In [ ]:
invoke_headers = {"Authorization": f"Bearer {AUTH_TOKEN}", "Content-Type": "application/json"}
session_id = str(uuid.uuid4())
print(f"session_id: {session_id}\n")
try:
    # Turn 1 — the agent answers and saves state to PostgreSQL under this session_id.
    print("Turn 1 — 'List my databases'")
    resp1 = requests.post(
        f"{agent_url}/invoke",
        headers=invoke_headers,
        json={"input": {"message": "List my databases", "session_id": session_id}, "streaming": False},
        timeout=120, verify=SSL_VERIFY,
    )
    print(f"Status: {resp1.status_code}")
    if resp1.ok:
        b1 = resp1.json()
        print(json.dumps(b1.get("output", b1), indent=2, default=str)[:1500])
    else:
        print(resp1.text[:500])

    # Turn 2 — same session_id; the agent should reload state and resolve "the first one".
    print("\nTurn 2 — 'Can you tell me more about the first one?'")
    resp2 = requests.post(
        f"{agent_url}/invoke",
        headers=invoke_headers,
        json={"input": {"message": "Can you tell me more about the first one?", "session_id": session_id}, "streaming": False},
        timeout=120, verify=SSL_VERIFY,
    )
    print(f"Status: {resp2.status_code}")
    if resp2.ok:
        b2 = resp2.json()
        print(json.dumps(b2.get("output", b2), indent=2, default=str)[:1500])
    else:
        print(resp2.text[:500])

    print("\nIf Turn 2 referred to the database from Turn 1, session memory is working.")
except Exception as e:
    print(f"Memory test failed: {e}")

---

## AgentO11y — Observe & Evaluate the Agent (the focus)

> Everything from here on uses the **`agento11y` Observability SDK**. Now that an agent is deployed and producing traces, the next sections **discover its project, read its traces and spans, run evaluations, record feedback, and read aggregate metrics** — the core of what AgentO11y offers.

## 7. Observability Projects

The deployment in Section 4 (created with `in_house_observability=True`) auto-creates an
observability **project whose name is the deployment id (`DEPLOYMENT_ID`)**. We list the
available projects and **select that one** so every observability and evaluation step
below operates on the traces your deployed agent just produced. Otherwise it falls back to
the `Evaluation Demo` project, then the first available one.

**SDK:** `projects_api.list()` / `projects_api.get(id=...)`


In [ ]:
DEMO_PROJECT = "Evaluation Demo"

# List all observability projects.
projects_resp = projects_api.list()
projects_dict = to_dict(projects_resp)
project_list = [p if isinstance(p, dict) else to_dict(p) for p in projects_dict.get("projects", [])]
print(f"Found {len(project_list)} project(s):\n")
for p in project_list:
    print(f"  - {p.get('name', 'N/A')}  (id: {p.get('id', 'N/A')})")

# Prefer the deployed agent's observability project (its name == DEPLOYMENT_ID) so the rest
# of the notebook inspects the traces produced in Sections 5-6, then "Evaluation Demo",
# then the first available project.
project_id = next((p.get("id") for p in project_list
                   if DEPLOYMENT_ID and p.get("name") == DEPLOYMENT_ID), None)
project_id = project_id or next((p.get("id") for p in project_list if p.get("name") == DEMO_PROJECT), None)
project_id = project_id or project_list[0].get("id")

# Set as default on the client so subsequent calls auto-inject it.
# Assigned directly; set_project_id() can fail in some teradataml builds.
client.project_id = project_id
print(f"\nSelected project_id: {project_id}")

# The detail endpoint can return 500 on the first call; the project still exists.
try:
    detail = projects_api.get(id=project_id)
    show_output("Project Details", detail)
except Exception as ge:
    print(f"(Project detail fetch failed, continuing: {ge})")

## 8. (Optional) Generate Additional Traces

Sections 5–6 already produced **real traces** by invoking the deployed agent — that is the
notebook's only trace source. This **optional** section simply sends a few **more** messages
to the same deployed `agent_url`, giving the observability and evaluation sections below a
richer set of traces to work with.

Every trace follows the same path as before — **deployed agent → `/invoke` → server-side
execution → Opik trace** — and lands in the agent's observability project (the one selected
in Section 7). Nothing is generated locally.

> **Skip this** if the traces from Sections 5–6 are enough. Requires an active deployment
> (`agent_url` from Section 4).


In [ ]:
invoke_headers = {"Authorization": f"Bearer {AUTH_TOKEN}", "Content-Type": "application/json"}

# Additional natural-language messages. Each call executes server-side inside the
# deployed, Opik-instrumented agent and emits a trace into its observability project.
extra_messages = [
    "What tables are in my first database?",
    "Summarize what you can help me with.",
    "Show me the row count for one of my tables.",
    "What columns does that table have?",
]

succeeded = 0
for i, msg in enumerate(extra_messages, 1):
    payload = {"input": {"message": msg}, "streaming": False}
    try:
        r = requests.post(
            f"{agent_url}/invoke",
            headers=invoke_headers,
            json=payload,
            timeout=120,
            verify=SSL_VERIFY,
        )
        print(f"  [{i}/{len(extra_messages)}] {msg!r} -> {'OK' if r.ok else f'HTTP {r.status_code}'}")
        if r.ok:
            succeeded += 1
        else:
            print(f"        {r.text[:300]}")
    except Exception as e:
        print(f"  [{i}/{len(extra_messages)}] {msg!r} -> failed: {type(e).__name__}: {e}")

# Give the observability backend a moment to ingest and index the new traces.
time.sleep(2)
print(f"\nGenerated {succeeded} additional trace(s) by invoking the deployed agent.")
print("They land in the observability project selected in Section 7 — retrieve them in Section 9.")

## 9. Search & View Traces

The first code cell searches the selected project for recent traces and keeps their IDs; the next one opens a single trace to inspect its detail and the spans (individual LLM and tool steps) it contains.

**SDK:** `traces_api.search(...)` / `traces_api.get(id=...)` / `traces_api.get_spans(id=...)`


In [12]:
trace_id = None  # will be set below for use in later sections
all_trace_ids: list[str] = []

try:
    # Search recent traces in the selected project (up to 20)
    traces_resp = traces_api.search(project_id=project_id, limit=20)
    traces_dict = to_dict(traces_resp)
    traces_list = traces_dict.get("traces", [])
    pagination = traces_dict.get("pagination", {})

    print("=== Traces (showing up to 20) ===")
    print(f"Total available: {pagination.get('total', 'unknown')}\n")

    for t in traces_list:
        t_d = to_dict(t) if not isinstance(t, dict) else t
        print(f"  Trace ID : {t_d.get('id', 'N/A')}")
        print(f"  Name     : {t_d.get('name', 'N/A')}")
        print(f"  Status   : {t_d.get('status', 'N/A')}")
        print(f"  Created  : {t_d.get('created_at', 'N/A')}")
        print()

    if traces_list:
        first = traces_list[0]
        trace_id = first.get("id") if isinstance(first, dict) else getattr(first, "id", None)
        all_trace_ids = [
            (t.get("id") if isinstance(t, dict) else getattr(t, "id", None))
            for t in traces_list
        ]
        print(f"Using trace_id = {trace_id} for single-trace sections.")
        print(f"Collected {len(all_trace_ids)} trace IDs for batch evaluation.")

except Exception as e:
    print(f"Trace search failed: {e}")


=== Traces (showing up to 20) ===
Total available: 2

  Trace ID : 019eeee2-b482-722b-a00a-0d8ca5de27ce
  Name     : LangGraph
  Status   : success
  Created  : 2026-06-22T10:31:37.239953Z

  Trace ID : 019eeee2-b395-7b2b-b441-31cd1eb88606
  Name     : LangGraph
  Status   : success
  Created  : 2026-06-22T10:31:37.239953Z

Using trace_id = 019eeee2-b482-722b-a00a-0d8ca5de27ce for single-trace sections.
Collected 2 trace IDs for batch evaluation.


**Drill into a single trace.** Using the `trace_id` captured above, fetch that trace's full detail and list its spans — the individual LLM and tool steps that make up the run.

In [13]:
try:
    # Fetch the full trace detail
    trace_detail = traces_api.get(id=trace_id, project_id=project_id)
    print("=== Trace Detail ===")
    show_output("Trace", trace_detail)
    print()

    # List the spans (LLM/tool steps) in this trace
    spans_resp = traces_api.get_spans(id=trace_id, project_id=project_id)
    spans_dict = to_dict(spans_resp)
    span_list = spans_dict.get("spans", [])

    print(f"=== Spans for trace {trace_id} ({len(span_list)} span(s)) ===")
    for s in span_list:
        s_d = to_dict(s) if not isinstance(s, dict) else s
        print(f"  Span ID  : {s_d.get('id', 'N/A')}")
        print(f"  Name     : {s_d.get('name', 'N/A')}")
        print(f"  Type     : {s_d.get('type', 'N/A')}")
        print()
except Exception as e:
    print(f"Trace detail request failed: {e}")

=== Trace Detail ===

Trace:
{'id': '019eeee2-b482-722b-a00a-0d8ca5de27ce',
 'project_id': '019eeee2-bbd1-742b-a837-f8aeb0657b02',
 'name': 'LangGraph',
 'status': 'success',
 'created_at': '2026-06-22T10:31:37.239953Z',
 'duration_ms': 48,
 'input': {'messages': [{'role': 'user', 'content': 'What is the weather like in New York today?'}],
           'session_id': 'invoke-session-047c865e'},
 'output': None,
 'metadata': {'created_from': 'langchain', 'thread_id': 'invoke-session-047c865e', 'ls_integration': 'langgraph'},
 'tokens': {'prompt': 0, 'completion': 0, 'total': 0},
 'cost_usd': 0.0,
 'span_count': 2,
 'feedback_scores': {},
 'spans': [],
 'evaluation_results': []}

=== Spans for trace 019eeee2-b482-722b-a00a-0d8ca5de27ce (2 span(s)) ===
  Span ID  : 019eeee2-b48a-74f8-9747-983f07259345
  Name     : ChatOpenAI
  Type     : llm

  Span ID  : 019eeee2-b488-77ef-a77f-a91181ff5c14
  Name     : agent
  Type     : other



## 9b. List All Spans for a Project

The code cell lists spans directly at the project level and filters them by type (LLM, tool) and status (errors) — handy for spotting slow or failing steps across every trace.

**SDK:** `spans_api.list(project_id=..., type=..., status=...)`


In [14]:
try:
    # List all spans for the project
    all_spans = spans_api.list(project_id=project_id, limit=20)
    all_spans_dict = to_dict(all_spans)
    span_list = all_spans_dict.get("spans", [])
    pagination = all_spans_dict.get("pagination", {})

    print(f"=== All Spans for project {project_id} ===")
    print(f"Returned: {len(span_list)} | has_more: {pagination.get('has_more', False)}\n")

    for s in span_list:
        s_d = to_dict(s) if not isinstance(s, dict) else s
        print(f"  Span ID  : {s_d.get('id', 'N/A')}")
        print(f"  Name     : {s_d.get('name', 'N/A')}")
        print(f"  Type     : {s_d.get('type', 'N/A')}")
        print(f"  Status   : {s_d.get('status', 'N/A')}")
        print(f"  Duration : {s_d.get('duration_ms', 0)}ms")
        print()

    # Filter by type — only LLM spans (using SpanType enum)
    llm_spans_resp = spans_api.list(project_id=project_id, type=SpanType.llm, limit=10)
    llm_dict = to_dict(llm_spans_resp)
    llm_spans = llm_dict.get("spans", [])
    print(f"=== LLM Spans Only: {len(llm_spans)} span(s) ===")
    for s in llm_spans:
        s_d = to_dict(s) if not isinstance(s, dict) else s
        model = s_d.get("model", "unknown")
        tokens = s_d.get("tokens", {})
        total = tokens.get("total", 0) if isinstance(tokens, dict) else 0
        print(f"  {s_d.get('id')} | {s_d.get('name')} | model={model} | tokens={total}")

    # Filter by type — only tool spans
    tool_spans_resp = spans_api.list(project_id=project_id, type=SpanType.tool, limit=10)
    tool_dict = to_dict(tool_spans_resp)
    tool_spans = tool_dict.get("spans", [])
    print(f"\n=== Tool Spans: {len(tool_spans)} span(s) ===")
    for s in tool_spans:
        s_d = to_dict(s) if not isinstance(s, dict) else s
        print(f"  {s_d.get('id')} | {s_d.get('name')} | duration={s_d.get('duration_ms', 0)}ms")

    # Filter by status — only errors (using TraceStatus enum)
    err_resp = spans_api.list(project_id=project_id, status=TraceStatus.error, limit=10)
    err_dict = to_dict(err_resp)
    err_spans = err_dict.get("spans", [])
    print(f"\n=== Error Spans: {len(err_spans)} span(s) ===")
    for s in err_spans:
        s_d = to_dict(s) if not isinstance(s, dict) else s
        print(f"  {s_d.get('id')} | {s_d.get('error_type', 'N/A')}: {s_d.get('error_message', 'N/A')}")
except Exception as e:
    print(f"Spans request failed: {e}")

=== All Spans for project 9c432b48-dd15-4008-86d8-b747aa32273d ===
Returned: 4 | has_more: False

  Span ID  : 019eeee2-b48a-74f8-9747-983f07259345
  Name     : ChatOpenAI
  Type     : llm
  Status   : success
  Duration : 29ms

  Span ID  : 019eeee2-b488-77ef-a77f-a91181ff5c14
  Name     : agent
  Type     : other
  Status   : success
  Duration : 35ms

  Span ID  : 019eeee2-b3c0-70b1-a75d-d9dd9c699e89
  Name     : ChatOpenAI
  Type     : llm
  Status   : success
  Duration : 75ms

  Span ID  : 019eeee2-b3be-77a8-87ec-bc4394ed90fb
  Name     : agent
  Type     : other
  Status   : success
  Duration : 80ms

=== LLM Spans Only: 2 span(s) ===
  019eeee2-b48a-74f8-9747-983f07259345 | ChatOpenAI | model=None | tokens=0
  019eeee2-b3c0-70b1-a75d-d9dd9c699e89 | ChatOpenAI | model=None | tokens=0

=== Tool Spans: 0 span(s) ===

=== Error Spans: 0 span(s) ===


## 10. Evaluation Rules

Evaluation rules score traces automatically. The cells below confirm an LLM API key is available, list any existing rules, then create an LLM-judge rule that rates response quality.

**SDK:** `evaluations_api.list_rules(...)` / `evaluations_api.create_rule(body=...)`


In [15]:
# LLM-based evaluations (Sections 10-11, 15) use this key; it was entered in the Section 1
# credentials cell. The server can also supply AGENT_O11Y_LLM_API_KEY / OPENAI_API_KEY.
openai_key = os.environ.get("OPENAI_API_KEY", "")
print(f"OPENAI_API_KEY set: {bool(openai_key)}")

OPENAI_API_KEY set: True


**List existing rules.** Show any evaluation rules already defined on the selected project before adding a new one.

In [16]:
rule_id = None  # will be set if a rule is created

try:
    # List evaluation rules on the project
    rules_resp = evaluations_api.list_rules(project_id=project_id)
    rules_dict = to_dict(rules_resp)
    rules = rules_dict.get("rules", [])
    print(f"=== Existing Evaluation Rules ({len(rules)}) ===")
    for r in rules:
        r_d = to_dict(r) if not isinstance(r, dict) else r
        print(f"  - {r_d.get('name', 'N/A')} (id: {r_d.get('id', 'N/A')}, category: {r_d.get('category', 'N/A')})")
    if not rules:
        print("  (none found)")
except Exception as e:
    print(f"List rules failed: {e}")


=== Existing Evaluation Rules (2) ===
  - Response Quality Judge (id: 019eee53-8e9d-7169-b333-69906261da6e, category: quality)
  - Response Quality Judge (id: 019eee28-5abc-702f-9131-89cf9d2d038f, category: quality)


**Create an LLM-judge rule.** Define a rule that asks `gpt-4o-mini` to score each response's quality from 0 to 1, built from the typed `EvaluationRuleCreate` model.

In [17]:
# Create a new LLM judge rule
try:
    # Create the LLM-judge rule
    rule_body = EvaluationRuleCreate(
        name="Response Quality Judge",
        description="Evaluates the quality and helpfulness of agent responses.",
        category=EvalCategory.quality,
        type=EvalType.llm_judge,
        model="gpt-4o-mini",
        enabled=True,
        sampling_rate=0.5,
        project_id=project_id,
        prompt_template=(
            "You are an evaluation judge. Assess the quality of the following "
            "agent response.\n\n"
            "User Input: {{input}}\n"
            "Agent Output: {{output}}\n\n"
            "Rate the response on a scale from 0.0 to 1.0 based on:\n"
            "- Relevance to the user's question\n"
            "- Completeness of the answer\n"
            "- Accuracy of information\n\n"
            "Return a JSON object with 'score' (float) and 'reason' (string)."
        ),
        # Typed models instead of raw dicts — they serialize identically.
        variable_mappings=[
            VariableMapping(variable="input", trace_field="input"),
            VariableMapping(variable="output", trace_field="output"),
        ],
        score_definitions=[
            ScoreDefinition(
                name="response_quality",
                type=ScoreType.decimal,
                description="Overall quality score from 0 to 1",
            )
        ],
    )

    created_rule = evaluations_api.create_rule(body=rule_body)
    created_dict = to_dict(created_rule)
    rule_id = created_dict.get("id")

    show_output("Created Evaluation Rule", created_rule)
    print(f"\nRule ID: {rule_id}")
except Exception as e:
    print(f"Create rule failed: {e}")


Created Evaluation Rule:
{'id': '019ef06f-25d7-736d-aeb5-09b095336854',
 'name': 'Response Quality Judge',
 'description': 'Evaluates the quality and helpfulness of agent responses.',
 'category': 'quality',
 'type': 'llm_judge',
 'model': 'gpt-4o-mini',
 'enabled': True,
 'sampling_rate': 0.5,
 'project_id': '9c432b48-dd15-4008-86d8-b747aa32273d',
 'created_at': '2026-06-22T17:44:36.648021Z'}

Rule ID: 019ef06f-25d7-736d-aeb5-09b095336854


## 11. Trigger Online Evaluation

The code cell runs the LLM-judge rule against the collected traces — server-side first, falling back to client-side — and the next one prints the resulting scores.

**SDK:** `evaluations_api.run(body=...)` / `evaluations_api.poll(id=..., project_id=...)`


In [18]:
eval_results = None  # populated by whichever path succeeds
eval_trace_ids = all_trace_ids if all_trace_ids else ([trace_id] if trace_id else [])

# ------------------------------------------------------------------
# Step 1: Trigger server-side async evaluation
# ------------------------------------------------------------------
try:
    eval_body = EvalJobRequest(
        trace_ids=eval_trace_ids,
        rule_ids=[rule_id],
        **{"async": True},  # 'async' is a Python keyword
    )
    print(f"[Step 1] Triggering server-side async evaluation for {len(eval_trace_ids)} trace(s)...")
    eval_resp = evaluations_api.run(body=eval_body, project_id=project_id)
    eval_resp_dict = to_dict(eval_resp)
    job_id = eval_resp_dict.get("job_id")
    print(f"  Job ID : {job_id}")
    print(f"  Status : {eval_resp_dict.get('status')}")
except Exception as e:
    print(f"  Server-side trigger failed: {e}")
    job_id = None

# ------------------------------------------------------------------
# Step 2: Poll the evaluation job until it finishes
# ------------------------------------------------------------------
if job_id:
    print(f"\n[Step 2] Using evaluations_api.poll() for job {job_id}...")
    # poll() checks job status every 5s for up to 60s.
    poll_resp = evaluations_api.poll(id=job_id, project_id=project_id)
    poll_dict = to_dict(poll_resp)
    status = poll_dict.get("status", "unknown")
    results = poll_dict.get("results") or []
    if status == "completed" and results:
        eval_results = results
        print(f"  -> Server-side scores received! ({len(eval_results)} trace(s))")
    elif status == "failed":
        print(f"  -> Job failed: {poll_dict.get('error_message')}")

# ------------------------------------------------------------------
# Step 3: Fallback — client-side sync evaluation
# ------------------------------------------------------------------
if not eval_results:
    print("\n[Step 3] No server-side scores within timeout. Running client-side sync evaluation...")
    try:
        fb_body = EvalJobRequest(
            trace_ids=eval_trace_ids,
            rule_ids=[rule_id],
            **{"async": False},
        )
        fb_resp = evaluations_api.run(body=fb_body, project_id=project_id)
        fb_dict = to_dict(fb_resp)
        eval_results = fb_dict.get("results") or []
        print(f"  Client-side evaluation complete: {len(eval_results)} trace(s) scored")
    except Exception as e:
        print(f"  Client-side evaluation failed: {e}")

[Step 1] Triggering server-side async evaluation for 2 trace(s)...
  Job ID : eval_79f1e9755927
  Status : completed

[Step 2] Using evaluations_api.poll() for job eval_79f1e9755927...
Polling Evaluations (every 5s, max 12 attempts)...
  [1/12] Status: None
  [2/12] Status: None
  [3/12] Status: None
  [4/12] Status: None
  [5/12] Status: completed
  Done — status: completed
  -> Server-side scores received! (2 trace(s))


**Show the scores.** Print the per-trace scores and reasons returned by whichever evaluation path — server-side or client-side — succeeded above.

In [19]:
print(f"=== Evaluation Scores ({len(eval_results or [])} trace(s)) ===")
for r in (eval_results or []):
    r_d = to_dict(r) if not isinstance(r, dict) else r
    tid = r_d.get("trace_id", "?")
    scores = r_d.get("scores", {})
    if scores:
        for name, detail in scores.items():
            d = to_dict(detail) if not isinstance(detail, dict) else detail
            score_val = d.get("score", "?") if isinstance(d, dict) else d
            reason = d.get("reason", "") if isinstance(d, dict) else ""
            print(f"  [{tid[:16]}...]  {name}: {score_val:.2f}  - {reason[:80]}")
    else:
        print(f"  [{tid[:16]}...]  (no scores)")

=== Evaluation Scores (2 trace(s)) ===
  [019eeee2-b482-72...]  (no scores)
  [019eeee2-b395-7b...]  (no scores)


## 12. Feedback Scores

Feedback scores record quality judgments on traces. The cells below write scores to a single trace, write a batch across traces, then read the recorded scores back.

**SDK:** `feedback_api.write(trace_id=..., body=...)` / `feedback_api.write_batch(body=...)`


In [20]:
# Write feedback scores to a single trace. This endpoint takes a bare JSON array,
# so the items are plain dicts.
try:
    feedback_payload = [
        {
            "name": "user_satisfaction",
            "value": 0.9,
            "reason": "User reported the response was accurate and helpful.",
        },
        {
            "name": "factual_accuracy",
            "value": 0.85,
            "reason": "Response contained correct information with minor omissions.",
        },
    ]

    # Write feedback scores to a single trace
    feedback_resp = feedback_api.write(
        trace_id=trace_id,
        body=feedback_payload,
        project_id=project_id,
    )
    print(feedback_resp)
except Exception as e:
    print(f"Write feedback failed: {e}")

trace_id='019eeee2-b482-722b-a00a-0d8ca5de27ce' scores_written=2


**Write feedback in a batch.** Attach feedback scores to one or more traces in a single call using the typed `FeedbackScoreBatchRequest` model.

In [21]:
# Batch write feedback scores to multiple traces
try:
    # Write feedback scores to multiple traces in one call
    # Nested typed models — the parent FeedbackScoreBatchRequest serializes them.
    batch_body = FeedbackScoreBatchRequest(
        items=[
            FeedbackScoreItem(
                trace_id=trace_id,
                scores=[
                    FeedbackScore(
                        name="response_completeness",
                        value=0.75,
                        reason="Response addressed the main question but missed edge cases.",
                    )
                ],
            ),
        ]
    )
    batch_resp = feedback_api.write_batch(body=batch_body, project_id=project_id)
    show_output("Batch Feedback Response", batch_resp)
except Exception as e:
    print(f"Batch feedback failed: {e}")


Batch Feedback Response:
{'total_traces': 1,
 'total_scores_written': 1,
 'results': [{'trace_id': '019eeee2-b482-722b-a00a-0d8ca5de27ce', 'scores_written': 1}],
 'errors': []}


**Read the scores back.** Retrieve the evaluation scores recorded on the project to confirm the feedback and evaluations above were stored.

In [22]:
# Read back evaluation scores
try:
    # Read back evaluation scores recorded on the project
    scores_resp = evaluations_api.get_scores(project_id=project_id, limit=10)
    scores_dict = to_dict(scores_resp)
    scores = scores_dict.get("scores", [])

    print(f"=== Evaluation Scores ({len(scores)} returned) ===")
    for s in scores:
        s_d = to_dict(s) if not isinstance(s, dict) else s
        print(
            f"  Trace: {s_d.get('trace_id', 'N/A')[:16]}...  "
            f"Rule: {s_d.get('rule_name', 'N/A')}  "
            f"Score: {s_d.get('score', 'N/A')}"
        )
    if not scores:
        print("  (no scores found)")

except Exception as e:
    print(f"Get scores failed: {e}")


=== Evaluation Scores (3 returned) ===
  Trace: 019eeee2-b482-72...  Rule: factual_accuracy  Score: 0.85
  Trace: 019eeee2-b482-72...  Rule: response_completeness  Score: 0.75
  Trace: 019eeee2-b482-72...  Rule: user_satisfaction  Score: 0.9


## 13. Metrics Catalog

The cells below browse the built-in metrics the platform can compute, then filter the catalog down to the LLM-judge metrics and their parameters.

**SDK:** `evaluations_api.get_metrics_catalog(...)` / `evaluations_api.get_metrics_catalog(type=...)`


In [23]:
# Browse the full metrics catalog
try:
    catalog_resp = evaluations_api.get_metrics_catalog()
    catalog_dict = to_dict(catalog_resp)
    metrics_list = catalog_dict.get("metrics", [])
    total = catalog_dict.get("total", 0)
    types = catalog_dict.get("types", {})

    print(f"=== Metrics Catalog ({total} total) ===")
    print("\nMetrics by type:")
    for metric_type, count in types.items():
        print(f"  {metric_type}: {count}")

    print("\nAll metrics:")
    for m in metrics_list:
        m_d = to_dict(m) if not isinstance(m, dict) else m
        print(f"  - {m_d.get('name', 'N/A')} ({m_d.get('type', 'N/A')}): {m_d.get('description', 'N/A')[:80]}")

except Exception as e:
    print(f"Catalog request failed: {e}")


=== Metrics Catalog (24 total) ===

Metrics by type:
  heuristic: 10
  llm_judge: 11
  conversation: 3

All metrics:
  - Equals (heuristic): Checks if output exactly matches the expected output.
  - Contains (heuristic): Checks if the output contains a specific substring.
  - RegexMatch (heuristic): Checks if the output matches a regular expression pattern.
  - IsJson (heuristic): Checks if the output is valid JSON.
  - LevenshteinRatio (heuristic): Calculates the Levenshtein distance ratio between output and expected output.
  - SentenceBLEU (heuristic): Calculates sentence-level BLEU score comparing output to reference.
  - CorpusBLEU (heuristic): Calculates corpus-level BLEU score comparing outputs to references.
  - ROUGE (heuristic): Calculates ROUGE scores (ROUGE-1, ROUGE-2, ROUGE-L) for text summarization quali
  - BERTScore (heuristic): Uses BERT embeddings for semantic similarity between output and reference.
  - Sentiment (heuristic): Analyzes the sentiment polarity of the ou

**Filter the catalog.** Narrow the metrics catalog to just the `llm_judge` metrics and show each one's parameters.

In [24]:
# Filter catalog by type: llm_judge
try:
    llm_judge_resp = evaluations_api.get_metrics_catalog(type="llm_judge")
    llm_dict = to_dict(llm_judge_resp)
    llm_metrics = llm_dict.get("metrics", [])

    print(f"=== LLM Judge Metrics ({len(llm_metrics)}) ===")
    for m in llm_metrics:
        m_d = to_dict(m) if not isinstance(m, dict) else m
        print(f"\n  Name       : {m_d.get('name', 'N/A')}")
        print(f"  Description: {m_d.get('description', 'N/A')}")
        params_list = m_d.get("parameters", [])
        if params_list:
            print("  Parameters :")
            for p in params_list:
                p_d = to_dict(p) if not isinstance(p, dict) else p
                req = "required" if p_d.get("required") else "optional"
                print(f"    - {p_d.get('name', '?')} ({p_d.get('type', '?')}, {req}): {p_d.get('description', '')}")

except Exception as e:
    print(f"Filtered catalog request failed: {e}")


=== LLM Judge Metrics (11) ===

  Name       : Hallucination
  Description: Detects hallucinated content by comparing output against provided context. Returns a score where 0 = no hallucination and 1 = fully hallucinated.
  Parameters :
    - model (str, optional): LLM model for evaluation (e.g., 'gpt-4o-mini', 'anthropic/claude-3-sonnet')

  Name       : AnswerRelevance
  Description: Evaluates how relevant the output answer is to the input question. Score from 0 (irrelevant) to 1 (highly relevant).
  Parameters :
    - model (str, optional): LLM model for evaluation (e.g., 'gpt-4o-mini', 'anthropic/claude-3-sonnet')

  Name       : ContextPrecision
  Description: Measures how precisely the retrieved context matches what is needed to answer the question. High precision = no irrelevant context.
  Parameters :
    - model (str, optional): LLM model for evaluation (e.g., 'gpt-4o-mini', 'anthropic/claude-3-sonnet')

  Name       : ContextRecall
  Description: Measures how much of the rele

## 14. Dataset Management

The cells below create a dataset, add question / expected-answer items to it, and list them back. This dataset feeds the offline evaluation in the next section.

**SDK:** `datasets_api.create(body=...)` / `datasets_api.insert_items(id=..., body=...)` / `datasets_api.get_items(id=...)`


In [ ]:
dataset_id = None  # will be set below

try:
    # Create the dataset
    dataset_body = DatasetCreate(
        name="demo-qa-dataset",
        description="Demo Q&A dataset for offline evaluation",
    )

    dataset_resp = datasets_api.create(body=dataset_body, project_id=project_id)
    dataset_dict = to_dict(dataset_resp)
    dataset_id = dataset_dict.get("id")

    show_output("Dataset Created", dataset_resp)
except Exception as e:
    print(f"Create dataset failed: {e}")

**Add items to the dataset.** Insert a few question / expected-answer pairs that offline evaluation will later score against.

In [ ]:
# Insert items into the dataset. This endpoint takes a bare JSON array,
# so the items are plain dicts.
try:
    items_payload = [
        {
            "input": {"question": "What is the capital of France?"},
            "expected_output": {"answer": "The capital of France is Paris."},
            "metadata": {"difficulty": "easy", "category": "geography"},
        },
        {
            "input": {"question": "Explain quantum entanglement in simple terms."},
            "expected_output": {
                "answer": (
                    "Quantum entanglement is a phenomenon where two particles "
                    "become connected so that the state of one instantly affects "
                    "the other, regardless of distance."
                )
            },
            "metadata": {"difficulty": "medium", "category": "physics"},
        },
        {
            "input": {"question": "What are the three laws of thermodynamics?"},
            "expected_output": {
                "answer": (
                    "1) Energy cannot be created or destroyed. "
                    "2) Entropy of an isolated system always increases. "
                    "3) Entropy approaches zero as temperature approaches absolute zero."
                )
            },
            "metadata": {"difficulty": "medium", "category": "physics"},
        },
    ]

    # Insert items into the dataset
    items_resp = datasets_api.insert_items(
        id=dataset_id,
        body=items_payload,
        project_id=project_id,
    )
    show_output("Inserted Items", items_resp)
except Exception as e:
    print(f"Insert items failed: {e}")

**List the dataset items.** Read back the items just inserted to confirm they were stored.

In [ ]:
# List items in the dataset
try:
    # List the dataset items
    items_list_resp = datasets_api.get_items(id=dataset_id, project_id=project_id)
    items_dict = to_dict(items_list_resp)
    items = items_dict.get("items", [])
    total = items_dict.get("total", 0)

    print(f"=== Dataset Items ({total} total) ===")
    for item in items:
        i_d = to_dict(item) if not isinstance(item, dict) else item
        print(f"  ID      : {i_d.get('id', 'N/A')}")
        print(f"  Input   : {json.dumps(i_d.get('input', {}))[:80]}")
        expected = i_d.get("expected_output", {})
        print(f"  Expected: {json.dumps(expected)[:80] if expected else 'N/A'}")
        print()
except Exception as e:
    print(f"List items failed: {e}")

## 15. Offline Evaluation

The code cell scores a whole dataset against chosen metrics (offline evaluation), and the next one fetches the experiment's results and per-item breakdown.

**SDK:** `evaluations_api.run_offline(body=...)` / `evaluations_api.get_offline_results(name=...)`


In [ ]:
experiment_name = None  # will be set below

try:
    # Run offline evaluation over the dataset
    # Typed metric models instead of dicts — nested inside OfflineEvalRequest.
    offline_body = OfflineEvalRequest(
        dataset_id=dataset_id,
        metrics=[
            OfflineMetric(metric_name="AnswerRelevance", model="gpt-4o-mini"),
            OfflineMetric(metric_name="Hallucination", model="gpt-4o-mini"),
        ],
        experiment_name="demo-offline-eval",
        task_type=TaskType.passthrough,
        llm_api_key=os.environ.get("OPENAI_API_KEY", ""),
        project_id=project_id,
    )

    offline_resp = evaluations_api.run_offline(body=offline_body)
    offline_dict = to_dict(offline_resp)
    experiment_name = offline_dict.get("experiment_name")

    show_output("Offline Evaluation Triggered", offline_resp)
except Exception as e:
    print(f"Offline evaluation failed: {e}")

**Fetch the results.** Retrieve the metric scores for the experiment triggered above, including the per-item breakdown.

In [ ]:
# Get offline evaluation results
try:
    # Give the evaluation time to process
    time.sleep(3)

    # Fetch the offline evaluation results
    results_resp = evaluations_api.get_offline_results(
        name=experiment_name,
        project_id=project_id,
    )
    results_dict = to_dict(results_resp)

    print("=== Offline Evaluation Results ===")
    print(f"  Experiment : {results_dict.get('experiment_name', 'N/A')}")
    print(f"  Status     : {results_dict.get('status', 'N/A')}")
    print(f"  Total items: {results_dict.get('total_items', 'N/A')}")

    summary = results_dict.get("metric_scores_summary", {})
    if summary:
        print("\n  Metric Score Summary:")
        for metric, score in summary.items():
            print(f"    {metric}: {score:.4f}")

    item_results = results_dict.get("item_results", [])
    if item_results:
        print(f"\n  Per-Item Results ({len(item_results)} items):")
        for ir in item_results[:5]:
            ir_d = to_dict(ir) if not isinstance(ir, dict) else ir
            print(f"    Item: {ir_d.get('dataset_item_id', 'N/A')[:16]}...")
            for metric, score in ir_d.get("scores", {}).items():
                print(f"      {metric}: {score}")
except Exception as e:
    print(f"Get results failed: {e}")

## 16. Metrics Dashboard

The code cell pulls aggregate dashboard metrics for the project — token usage, cost, latency, errors, evaluation scores, and an overall summary.

**SDK:** `metrics_api.get_tokens()` / `get_costs()` / `get_latency()` / `get_errors()` / `get_evals()` / `get_summary()`


In [30]:
# Pull each aggregate dashboard metric for the project
metric_calls = [
    ("get_tokens", "Token Usage"),
    ("get_costs", "Costs"),
    ("get_latency", "Latency"),
    ("get_errors", "Errors"),
    ("get_evals", "Evaluation Scores"),
    ("get_summary", "Dashboard Summary"),
]

for method_name, label in metric_calls:
    print(f"{'=' * 50}")
    print(f"=== {label} (metrics_api.{method_name}) ===")
    print(f"{'=' * 50}")
    try:
        method = getattr(metrics_api, method_name)
        resp = method(project_id=project_id)
        show_output(label, resp)
    except Exception as e:
        print(f"  Request failed: {e}")
    print()


=== Token Usage (metrics_api.get_tokens) ===

Token Usage:
{'metrics': {'total': {'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 0},
             'average_per_trace': {'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 0},
             'time_series': [{'timestamp': '2026-06-22T00:00:00Z',
                              'tokens': {'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 0},
                              'trace_count': 4}],
             'by_model': {'unknown': {'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 0}}},
 'project_id': '9c432b48-dd15-4008-86d8-b747aa32273d',
 'time_range': {'start_time': None, 'end_time': None, 'granularity': 'day'}}

=== Costs (metrics_api.get_costs) ===

Costs:
{'metrics': {'total': {'prompt_cost': 0.0, 'completion_cost': 0.0, 'total_cost': 0.0, 'currency': 'USD'},
             'average_per_trace': 0.0,
             'time_series': [{'timestamp': '2026-06-22T00:00:00Z',
                              'cost': 

## 17. (Optional) Retire the Agent

This final step mirrors the "retire" stage of the agent lifecycle and is **destructive** —
it tears down the deployment whose traces the observability sections analyze:

1. **Delete the deployment** — stops the running agent and frees its resources.
2. **Archive the agent definition** — removes it from the active catalog.

Both steps are guarded by prerequisite checks and a `CONFIRM_CLEANUP` flag. It defaults to
`True` so a single **Run All** completes the entire lifecycle (build → … → retire). Set
`CONFIRM_CLEANUP = False` to keep the deployment running and explore it further.


In [31]:
# Safety switch — set to True only when you intend to tear down what you created.
CONFIRM_CLEANUP = True

if not CONFIRM_CLEANUP:
    print("Cleanup skipped. Set CONFIRM_CLEANUP = True to delete the deployment and archive the definition.")
else:
    # 1) Delete the deployment (stops the running agent and frees resources).
    try:
        show_output("Delete Deployment", deployments.delete(id=DEPLOYMENT_ID))
        print(f"Deleted deployment {DEPLOYMENT_ID}")
    except Exception as e:
        print(f"Delete deployment failed: {e}")

    # 2) Archive the agent definition (removes it from the active catalog).
    try:
        show_output("Archive Agent Definition", agent_definitions.archive(id=AGENT_DEF_ID))
        print(f"Archived agent definition {AGENT_DEF_ID}")
    except Exception as e:
        print(f"Archive agent definition failed: {e}")


Delete Deployment:
{'deployment_id': '01e67f83-d5d5-4510-9eaa-534258272ee0',
 'status': 'deleted',
 'message': 'Deployment deleted successfully',
 'deployment_name': 'agentops-o11y-demo-b78112df-01e67f83',
 'namespace': 'agentruntime'}
Deleted deployment 01e67f83-d5d5-4510-9eaa-534258272ee0

Archive Agent Definition:
{'id': 'fff652ad-ff26-4da4-a749-8bfd4ca75afc', 'message': "Agent 'agentops-o11y-demo-agent-b78112df' archived"}
Archived agent definition fff652ad-ff26-4da4-a749-8bfd4ca75afc


## 18. Summary & Next Steps

This notebook is an **AgentO11y (Observability) walkthrough**: it **observed** and **evaluated** a
live agent with the `agento11y` SDK — projects, traces & spans, evaluation rules, online/offline
evaluation, feedback scores, and metrics. To give it a real agent to watch, it first used the
`agentops` (Core) SDK as a **prerequisite** to **build, deploy, invoke, and add memory** to that agent.

### AgentO11y SDK — Observe & Evaluate (Sections 2, 7–16) — the focus

| Section | SDK Call |
|---------|----------|
| 2 | `health_api.health_check()` |
| 2 | `health_api.readiness()` |
| 7 | `projects_api.list()` |
| 7 | `projects_api.get(id=...)` |
| 9 | `traces_api.search(...)` |
| 9 | `traces_api.get(id=...)` |
| 9 | `traces_api.get_spans(id=...)` |
| 9b | `spans_api.list(...)` |
| 10 | `evaluations_api.list_rules(...)` |
| 10 | `evaluations_api.create_rule(body=...)` |
| 11 | `evaluations_api.run(body=...)` |
| 11 | `evaluations_api.poll(id=...)` |
| 12 | `feedback_api.write(trace_id=..., body=...)` |
| 12 | `feedback_api.write_batch(body=...)` |
| 12 | `evaluations_api.get_scores(...)` |
| 13 | `evaluations_api.get_metrics_catalog(...)` |
| 14 | `datasets_api.create(body=...)` |
| 14 | `datasets_api.insert_items(id=..., body=...)` |
| 14 | `datasets_api.get_items(id=...)` |
| 15 | `evaluations_api.run_offline(body=...)` |
| 15 | `evaluations_api.get_offline_results(name=...)` |
| 16 | `metrics_api.get_tokens(...)` |
| 16 | `metrics_api.get_costs(...)` |
| 16 | `metrics_api.get_latency(...)` |
| 16 | `metrics_api.get_errors(...)` |
| 16 | `metrics_api.get_evals(...)` |
| 16 | `metrics_api.get_summary(...)` |

### AgentOps Core SDK — Create the Agent to Observe (prerequisite, Sections 3–6, 17)

| Section | Step | SDK Call |
|---------|------|----------|
| 3 | Create agent definition | `agent_definitions.create(body=AgentDefinitionCreateRequest(...), file=ZIP_PATH)` |
| 3 | Wait for processing | `agent_definitions.poll(id=...)` |
| 3 | Validate definition | `agent_definitions.get(id=...)` / `agent_definitions.list(...)` |
| 4 | Create deployment | `deployments.create(body=DeploymentCreateRequest(..., enable_memory=True, in_house_observability=True))` |
| 4 | Wait for active | `deployments.poll(id=...)` |
| 4 | Inspect deployment | `deployments.get(id=...)` (`agent_url`) / `deployments.logs(id=...)` |
| 5 | Invoke the agent | `POST {agent_url}/invoke` → `{"input": {"message": ...}, "streaming": False}` |
| 6 | Session memory test | multi-turn `/invoke` with a shared `session_id` |
| 17 | Retire the agent | `deployments.delete(id=...)` / `agent_definitions.archive(id=...)` |

### What we demonstrated

| Stage | SDK | Section |
|-------|-----|---------|
| Health checks | `agento11y.Health` / `agentops.Health` | 2 |
| Observability projects | `agento11y.Projects` | 7 |
| Generate additional traces (optional) | Deployed agent `/invoke` | 8 |
| Trace & span inspection | `agento11y.Traces` / `agento11y.Spans` | 9, 9b |
| Evaluation rules & online eval | `agento11y.Evaluations` | 10, 11 |
| Feedback scores | `agento11y.FeedbackScores` | 12 |
| Metrics catalog | `agento11y.Evaluations` | 13 |
| Dataset management | `agento11y.Datasets` | 14 |
| Offline evaluation | `agento11y.Evaluations` | 15 |
| Metrics dashboard | `agento11y.Metrics` | 16 |
| Build agent definition (prerequisite) | `agentops.Definitions` | 3 |
| Deploy agent (prerequisite) | `agentops.Deployments` | 4 |
| Invoke agent (prerequisite) | HTTP `/invoke` | 5 |
| Session memory (prerequisite) | `enable_memory=True` + `session_id` | 6 |
| Retire agent (optional prerequisite cleanup) | `agentops.Deployments` / `Definitions` | 17 |

### Next steps

- Re-run against your own agent by pointing the project selector at any project that already has traces (you can skip Sections 3–6).
- Add evaluation rules tailored to your use case (Section 10) and schedule online evaluation (Section 11).
- Build datasets from production traces and track quality over time with offline evaluation (Sections 14–15).
- Wire the Metrics Dashboard (Section 16) into a monitoring routine for tokens, cost, latency, and errors.


## 19. Data Models Reference

Use this quick reference for the request models and connection inputs used throughout the notebook.

### 19.1 Request Models Used in This Notebook

| SDK | Request Model(s) | Purpose |
|-----|------------------|---------|
| Core | `AgentDefinitionCreateRequest`, `ArtifactsConfig` | Register an agent definition from a ZIP (Section 3) |
| Core | `DeploymentCreateRequest`, `DeploymentEngineTypeConfig`, `DeploymentResourceConfig`, `DeploymentServiceConfig` | Deploy an agent (Section 4) |
| O11y | `EvaluationRuleCreate`, `VariableMapping`, `ScoreDefinition` | Create an LLM-judge evaluation rule (Section 10) |
| O11y | `EvalJobRequest` | Run online evaluation (Section 11) |
| O11y | `FeedbackScoreBatchRequest`, `FeedbackScoreItem`, `FeedbackScore` | Write feedback scores (Section 12) |
| O11y | `DatasetCreate` | Create a dataset (Section 14) |
| O11y | `OfflineEvalRequest`, `OfflineMetric` | Run offline evaluation (Section 15) |

### 19.2 Connection Inputs

All inputs are configured in step 1.3.

| Input | How it is provided | Description |
|-------|--------------------|-------------|
| `AUTH_TOKEN` | Prompted via `getpass` | JWT bearer token attached to every request — authenticates **both** services |
| `BASE_URL` | Prompted via `getpass` | AgentOps service base URL — fronts **both** the Core and O11y services |
| `OPENAI_API_KEY` | Prompted via `getpass` (optional) | Injected into the deployment runtime; also used for LLM-based evaluations |
| `SSL_VERIFY` | Set in code | Verify TLS certificates (defaults to `False`; set `True` for trusted certs) |

Last validated: 2026-06-23